# **🏢 Amazon ML Challenge 2026 - Business Entity Resolution**

This project focuses on matching business entity records across noisy, independent data sources with no shared unique IDs.

In this notebook:

# **🧹 01. Data Audit, Cleaning & Normalization Pipeline**

This notebook handles data ingestion, structural verification, noise distribution analysis, and text standardization across Source 1 (reference), Source 2, and Source 3 before candidate generation (blocking).

## **1. Problem Definition**

Business identity data arrives from disparate sources with typos, legal suffixes, missing street details, and varying regional address formats.

Given raw business records across 3 TSV files, normalize text components so that downstream blocking rules can group candidate pairs accurately without dropping true matches.



## **2. Data Architecture & Column Mapping**

All datasets are tab-separated (`.tsv`) to safely handle commas inside address strings.

**Key Datasets:**
- `train_source1.tsv`: Reference deduplicated records (`S1-` prefix)
- `train_source2.tsv`: Query/fragment source 1 (`S2-` prefix)
- `train_source3.tsv`: Query/fragment source 2 (`S3-` prefix)
- `train_ground_truth.tsv`: Ground truth mapping (`source1_entity_id` → matched `S2`/`S3` IDs)

**Schema Across All Source Files:**

| **Column** | **Type** | **Description** | **Common Noise / Challenges** |
|---|---|---|---|
| `entity_id` | String | Unique record ID (`S1-xxx`, `S2-xxx`, `S3-xxx`) | Source prefix indicator |
| `business_name` | String | Name of the business | Legal suffixes (LLC, Pvt Ltd, Inc), typos, word reordering |
| `business_address` | String | Street address / location | Missing values, landmark references, abbreviations (`Rd`, `St`) |
| `country` | String | Country code (`US`, `India` in Train; `France` in Test) | Domain shift across regions |



## **3. Pipeline Execution Steps**

1. TSV Integrity & Ingestion: Verify tab separation and memory usage.
2. Missingness & Country Coverage Audit: Check null percentages for names and addresses across US vs India.
3. String Normalization Engine: Lowercasing, punctuation stripping, whitespace collapsing, and legal suffix stripping.
4. Digit & Token Extraction: Isolate street numbers/zip codes and sorted name tokens for blocking keys.

```python
Raw TSV file
     |
     v
Step 1: Does it load correctly? (integrity check)
     |
     v
Step 2: What's missing, and does it differ by country?
     |
     v
Step 3: Clean text (same business should look the same)
     |
     v
Step 4: Pull out blocking ingredients (zip, number, name tokens)
     |
     v
Ready to actually build bucket logic
```

### **1. Loading and testing the dataset**

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set up data directory path
DATA_DIR = Path("../data/student_resource/dataset/train")

# ---------------------------------------------------------
# STEP 1: TSV Integrity & Ingestion
# ---------------------------------------------------------
print("Loading datasets...")

df_s1 = pd.read_csv(DATA_DIR / "train_source1.tsv", sep="\t")
df_s2 = pd.read_csv(DATA_DIR / "train_source2.tsv", sep="\t")
df_s3 = pd.read_csv(DATA_DIR / "train_source3.tsv", sep="\t")
df_gt = pd.read_csv(DATA_DIR / "train_ground_truth.tsv", sep="\t")

print("Ingestion complete!\n")
print("Ingestion2 complete!\n")

Loading datasets...
Ingestion complete!



In [ ]:
df_s1.head()

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [ ]:
df_s2.head()

,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


In [ ]:
df_s3.head()

,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India


In [ ]:
df_gt.head()

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [ ]:
df_s1.isnull().sum()

entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

In [ ]:
df_s2.isnull().sum()

entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64

In [ ]:
pct = (df_s2.isnull().sum() / len(df_s2)) * 100
pct

entity_id           0.000000
business_name       0.000040
business_address    3.356105
country             0.000000
dtype: float64

In [ ]:
df_s3.isnull().sum()

entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64

In [ ]:
df_gt.isnull().sum()

source1_entity_id          0
matched_entity_ids    123247
dtype: int64

In [ ]:
print(len(df_s1))
print(len(df_s2))
print(len(df_s3))
print(len(df_gt))

2206821
5034616
5285603
2206821


In [ ]:
# Function to calculate missingness summary
def audit_missingness(df, name):
    missing = df.isnull().sum()
    pct = (df.isnull().sum() / len(df)) * 100
    audit_df = pd.DataFrame({'Missing Count': missing, 'Missing %': pct.round(2)})
    print(f"--- Missingness Audit: {name} ---")
    print(audit_df)
    print("\nCountry Breakdown:")
    print(df['country'].value_counts(dropna=False))
    print("=" * 45 + "\n")

audit_missingness(df_s1, "Source 1 (S1)")
audit_missingness(df_s2, "Source 2 (S2)")
audit_missingness(df_s3, "Source 3 (S3)")

--- Missingness Audit: Source 1 (S1) ---
                  Missing Count  Missing %
entity_id                     0        0.0
business_name                 0        0.0
business_address              0        0.0
country                       0        0.0

Country Breakdown:
country
US       1323633
India     883188
Name: count, dtype: int64

--- Missingness Audit: Source 2 (S2) ---
                  Missing Count  Missing %
entity_id                     0       0.00
business_name                 2       0.00
business_address         168967       3.36
country                       0       0.00

Country Breakdown:
country
US       3016817
India    2017799
Name: count, dtype: int64

--- Missingness Audit: Source 3 (S3) ---
                  Missing Count  Missing %
entity_id                     0       0.00
business_name                13       0.00
business_address         175916       3.33
country                       0       0.00

Country Breakdown:
country
US       3170056
India    

## **📊 Amazon ML Challenge 2026 — Data Audit & Normalization Summary**

### **1. Quick Dataset Overview**

| File | Total Records | Missing Names | Missing Addresses | Missing % (Addr) | US Count | India Count |
|---|---|---|---|---|---|---|
| `train_source1.tsv` | 2,206,821 | 0 | 0 | **0.0%** | 1,323,633 | 883,188 |
| `train_source2.tsv` | 5,034,616 | 2 | 168,967 | **3.36%** | 3,016,817 | 2,017,799 |
| `train_source3.tsv` | 5,285,603 | 13 | 175,916 | **3.33%** | 3,170,056 | 2,115,547 |

---

### **2. Key Audit Insights**

1. **Source 1 is Complete (Ground Reference):**
   - $S_1$ has zero missing values across all columns. It acts as our clean reference anchor.
   
2. **The Address Missingness Trap (~344,000 Records):**
   - Over **344,000 records** across $S_2$ and $S_3$ have no `business_address`.
   - **Pipeline Rule:** We **cannot** use address-only blocking keys. Any candidate generation rule requiring an address will permanently drop 3.3%+ of potential matches, lowering our maximum achievable recall.

3. **Geographic Distribution:**
   - Training set ratio is roughly **60% US / 40% India**.
   - Test set will introduce **France**. All cleaning and blocking logic must remain language-agnostic and handle French formats seamlessly.

---

## **3. Standardized Cleaning & Feature Strategy**

To ensure matching works across all sources, we generate three standardized representations per record:

1. `clean_name`: Lowercased, stripped punctuation, and stripped legal suffixes (US, India, and France: `inc`, `llc`, `pvt`, `ltd`, `sarl`, `sas`, `sa`, etc.).
2. `sorted_name_tokens`: Alphabetically ordered word tokens to handle word reordering (e.g., *"Metro Shoes Downtown"* $\to$ *"downtown metro shoes"*).
3. `address_digits`: Extracted numbers/zips from address strings to serve as hard numeric anchors independent of text typos.

---

In [ ]:
import re
import pandas as pd
import unicodedata
import time

# Combined Legal Suffixes: English (US/India), French (Test set), and Hindi Transliterations
LEGAL_SUFFIXES = r'\b(pvt|ltd|inc|llc|corp|corporation|co|limited|private|gmbh|sa|sarl|sas|sasu|eurl|sci|snc|gie|प्राइवेट|लिमिटेड|एलएलपी)\b'

def strip_latin_accents(text: str) -> str:
    """Converts French/Latin accents to base characters (e.g., 'é' -> 'e', 'ç' -> 'c')
    while keeping non-Latin scripts (like Hindi Devanagari) unaffected.
    """
    # NFKD decomposition separates base letters from combining accent marks
    nfkd_form = unicodedata.normalize('NFKD', text)
    # Remove only the combining accent marks (\p{Mn})
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

def normalize_series(series: pd.Series) -> pd.Series:
    """Vectorized normalization that folds French accents and preserves Hindi script."""
    # 1. Fill NaNs
    s = series.fillna("")

    # 2. Lowercase
    s = s.str.lower()

    # 3. Strip French accents (e.g., 'société' -> 'societe') so accented & unaccented records match
    s = s.apply(strip_latin_accents)

    # 4. Strip punctuation BUT KEEP words (\w), spaces (\s), and Devanagari script (\u0900-\u097F)
    s = s.str.replace(r'[^\w\s\u0900-\u097F]', ' ', regex=True)

    # 5. Strip legal corporate suffixes (English, French, Hindi)
    s = s.str.replace(LEGAL_SUFFIXES, ' ', regex=True)

    # 6. Collapse multiple spaces and trim
    s = s.str.replace(r'\s+', ' ', regex=True).str.strip()

    return s

# Time the execution to ensure it doesn't hang
print("Normalizing Source 1...")
start = time.time()
df_s1['clean_name'] = normalize_series(df_s1['business_name'])
df_s1['clean_address'] = normalize_series(df_s1['business_address'])
print(f"Source 1 done in {time.time() - start:.2f} seconds\n")

print("Normalizing Source 2 (The 5M row beast)...")
start = time.time()
df_s2['clean_name'] = normalize_series(df_s2['business_name'])
df_s2['clean_address'] = normalize_series(df_s2['business_address'])
print(f"Source 2 done in {time.time() - start:.2f} seconds\n")

print("Normalizing Source 3...")
start = time.time()
df_s3['clean_name'] = normalize_series(df_s3['business_name'])
df_s3['clean_address'] = normalize_series(df_s3['business_address'])
print(f"Source 3 done in {time.time() - start:.2f} seconds")

Normalizing Source 1...
Source 1 done in 28.82 seconds

Normalizing Source 2 (The 5M row beast)...
Source 2 done in 68.24 seconds

Normalizing Source 3...
Source 3 done in 76.11 seconds


In [ ]:
df_s1.head()

,entity_id,business_name,business_address,country,clean_name,clean_address
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,orelee s barbershop,1795 westchester drive high point nc
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,prime money,17560 ellis road tahlequah ok
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,b retail,1712 montebello avenue phoenix az
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,christ chapel,2100 cameron drive unit apartment g dundalk md
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,prabhav business center,797 lake town block a kolkata howrah west bengal


In [ ]:
df_s2.head()

,entity_id,business_name,business_address,country,clean_name,clean_address
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,राम मारकेटिंग पराइवेट,kh no 570 13 new delhi west delhi delhi
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,holloway peak seafood,105 elm st morganton nc
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,आदितय परॉपरटीज एलएलपी,g 3 571 gulmohar colony bhopal madhya pradesh
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US,summit,greensboro nc 19 1 2 stardust trail
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,delta tetlecommunication,914 pierpont ave cleveland oh


In [ ]:
df_s3.head()

,entity_id,business_name,business_address,country,clean_name,clean_address
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US,wilfordhancock com,mack rd haltom city texas
1,S3-859268022,International South Consultants Private Ltd,NaN,India,international south consultants,
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US,moncada learning center,5780 fawn ct fort worth texas
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US,moyna s coffee,1 ivanhoe ave po box 6009 cincinnati ohio
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India,efs print ventures,door no 183 41st cross 22nd main 9th block jay...


In [ ]:
df_gt.head()

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [ ]:
import time

print("Saving cleaned datasets to Parquet format using fastparquet...")
start = time.time()

# We added engine='fastparquet' to bypass the PyArrow bug
df_s1.to_parquet("../data/clean_source1.parquet", index=False, engine='fastparquet')
print("Source 1 saved.")

df_s2.to_parquet("../data/clean_source2.parquet", index=False, engine='fastparquet')
print("Source 2 saved.")

df_s3.to_parquet("../data/clean_source3.parquet", index=False, engine='fastparquet')
print(f"Source 3 saved. All files exported in {time.time() - start:.2f} seconds.")

Saving cleaned datasets to Parquet format using fastparquet...
Source 1 saved.
Source 2 saved.
Source 3 saved. All files exported in 51.27 seconds.


## **🧹 Data Cleaning & Normalization Handoff**

The initial data ingestion and text normalization phase is complete. The datasets are cleaned, standardized, and ready for the candidate generation (blocking) phase.

### **1. What Was Added**
- **New Columns:** Added `clean_name` and `clean_address` to all three datasets ($S_1$, $S_2$, $S_3$).
- **Preserved Raw Data:** The original `business_name` and `business_address` columns remain untouched for reference or fallback.

### **2. What Was Removed & Why**
- **French/Latin Accents Folded:** Accents were converted to base letters (e.g., `Société` $\to$ `societe`) using Unicode decomposition. *Why:* Prevents matching failures on the France test set when sources inconsistently use accents.
- **Punctuation Stripped (Safely):** Removed commas, periods, and symbols, but strictly preserved English/French alphanumeric characters (`\w`) AND the entire Hindi Devanagari unicode block (`\u0900-\u097F`). *Why:* Standard regex destroys Hindi matras (vowels). This custom regex ensures Hindi text stays 100% intact.
- **Legal Corporate Suffixes Removed:** Stripped high-frequency legal fluff across all three regions (e.g., `inc`, `llc`, `pvt`, `ltd`, `sarl`, `sas`, `प्राइवेट`, `लिमिटेड`). *Why:* These suffixes are noise. Stripping them ensures the core brand name is isolated for exact-match blocking and similarity scoring.
- **Whitespace Collapsed:** Multiple spaces and trailing whitespaces were trimmed into single spaces.

### **3. How It Is Saved**
The cleaned DataFrames were exported to the `data/clean_data/` folder as **Parquet files**, not TSVs.
- `clean_source1.parquet`
- `clean_source2.parquet`
- `clean_source3.parquet`

*Why Parquet:* It reads/writes significantly faster than TSV, uses less disk space, and critically, it preserves empty strings `""` instead of accidentally converting them back into `NaN` values like pandas sometimes does with CSV/TSV.

### **4. How to Load It (For the Blocking Team)**
You can load these files instantly into your blocking script using `pandas`.

```python
import pandas as pd

# Load the pre-cleaned datasets
df_s1 = pd.read_parquet("../data/clean_data/clean_source1.parquet")
df_s2 = pd.read_parquet("../data/clean_data/clean_source2.parquet")
df_s3 = pd.read_parquet("../data/clean_data/clean_source3.parquet")

# Now you can use clean_name and clean_address to build your blocking keys
# (e.g., generating sorted tokens or extracting address digits)